# 実践マーケ-03. A/Bテストで意思決定する

**舞台設定**：申込ページのボタンの色を「青(A)」と「緑(B)」で出し分けた。
どちらが申込率が高いか、`ab_test.csv` で科学的に判断します。

**使う統計（3〜2級）**：比率の比較・カイ二乗検定・「差は偶然か？」の判断

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
plt.rcParams['axes.unicode_minus'] = False
ab = pd.read_csv('../data/ab_test.csv')
ab.head()

## 1. 各グループの申込率を出す

まず単純に「見た目の率」を比べます。

In [ ]:
summary = ab.groupby('グループ')['申込'].agg(['count','sum','mean'])
summary.columns = ['訪問数','申込数','申込率']
summary['申込率'] = (summary['申込率']*100).round(2)
summary

In [ ]:
summary['申込率'].plot(kind='bar', figsize=(5,4), title='ボタン色別の申込率(%)')
plt.ylabel('申込率(%)'); plt.xticks(rotation=0); plt.show()

## 2. その差は「偶然」ではない？

Bの方が高く見えても、たまたまかもしれません。
**カイ二乗検定**で「色と申込に関連があるか」を確かめます。

- H₀（帰無仮説）：色と申込率は関係ない（差は偶然）
- H₁（対立仮説）：色によって申込率が違う

In [ ]:
table = pd.crosstab(ab['グループ'], ab['申込'])
print(table)
chi2, p, dof, exp = stats.chi2_contingency(table)
print(f'\nカイ二乗={chi2:.3f}, p値={p:.4f}')
if p < 0.05:
    print('→ 有意差あり！ ボタンの色で申込率は本当に変わる')
else:
    print('→ 有意差なし。偶然の範囲かもしれない')

## 3. 効果の大きさと事業インパクト

「統計的に有意」だけでなく「ビジネスとして何円得か」も大事です。

In [ ]:
rate_a = summary.loc['A_青ボタン','申込率'] / 100
rate_b = summary.loc['B_緑ボタン','申込率'] / 100
lift = (rate_b - rate_a) / rate_a * 100
print(f'改善率（リフト）: {lift:+.1f}%')

# 月10万訪問・1申込5000円の価値と仮定したら？
monthly_visits = 100000
value_per = 5000
gain = monthly_visits * (rate_b - rate_a) * value_per
print(f'Bに変えると月あたり約 {gain:,.0f} 円の増加見込み')

> 📝 **A/Bテストの注意点**
> - 途中で何度も覗いて「有意になった瞬間にやめる」のはNG（偽陽性が増える）
> - 事前に必要なサンプル数を決めておく
> - 期間は曜日の偏りをならすため1週間以上が目安

---
## 🏆 ワークシート課題

**課題1.** AとBの申込率の差を「パーセントポイント」と「リフト%」の両方で表そう。

**課題2.** もし有意水準を 0.01（より厳しく）にしても結論は変わるか、p値を見て判断しよう。

**課題3.**（意思決定）上司に「緑ボタンを採用すべきか」を、
統計的根拠（p値）とビジネス的根拠（増加見込み金額）の両方で説明する文を書こう。

**課題4.**（発展）`proportions_ztest` を使って同じ検定を比率の検定で行い、
カイ二乗検定とp値がほぼ一致することを確かめよう。

In [ ]:
# 課題1


In [ ]:
# 課題4（ヒント）
from statsmodels.stats.proportion import proportions_ztest


<details><summary>解答例</summary>

```python
count = ab.groupby('グループ')['申込'].sum().values
nobs  = ab.groupby('グループ')['申込'].count().values
z, p = proportions_ztest(count, nobs)
print(z, p)   # カイ二乗のp値とほぼ一致
```
</details>

🎉 **実践マーケ編 クリア！** 統計が「ビジネスの意思決定の道具」だと実感できたはず。
ひと息ついたら `06_ホビー_陶芸3D` で 3Dモデリングを楽しもう。